# Project 5: Emergency Shelter Occupancy Predictor - Exploratory Data Analysis

This notebook explores the Calgary Emergency Shelters Daily Occupancy dataset
to understand occupancy patterns, temporal trends, and shelter-level differences
before building predictive models.

**Dataset**: Emergency Shelters Daily Occupancy (ID: `7u2t-3wxf`)  
**Source**: City of Calgary Open Data Portal  
**Records**: ~82,869

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add project root to path so we can import src modules
PROJECT_DIR = Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR))

from src.data_loader import fetch_data, preprocess, add_rolling_features, compute_shelter_summary

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

## 1. Load the Data

In [ ]:
# Fetch raw data (uses cache if available)
raw_df = fetch_data(use_cache=True)
print(f"Raw data shape: {raw_df.shape}")
raw_df.head()

In [ ]:
# Column names and types
raw_df.info()

In [ ]:
# Missing values
print("Missing values per column:")
print(raw_df.isnull().sum())
print(f"\nTotal missing cells: {raw_df.isnull().sum().sum()}")

## 2. Preprocess the Data

In [ ]:
df = preprocess(raw_df)
print(f"Preprocessed shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
df.head()

In [ ]:
# Summary statistics for numeric columns
df[["capacity", "overnight", "occupancy_rate"]].describe()

## 3. Occupancy Rate Distribution

In [ ]:
fig = px.histogram(
    df, x="occupancy_rate", nbins=80,
    title="Distribution of Daily Occupancy Rates",
    labels={"occupancy_rate": "Occupancy Rate", "count": "Frequency"},
    marginal="box",
)
fig.add_vline(x=0.9, line_dash="dash", line_color="red",
              annotation_text="90% threshold")
fig.update_layout(height=450)
fig.show()

In [ ]:
# What fraction of observations exceed 90% occupancy?
above_90 = (df["occupancy_rate"] > 0.9).mean()
above_100 = (df["occupancy_rate"] > 1.0).mean()
print(f"Fraction above 90% occupancy: {above_90:.1%}")
print(f"Fraction above 100% occupancy (overcapacity): {above_100:.1%}")

In [ ]:
fig = px.histogram(
    df, x="overnight", nbins=80,
    title="Distribution of Overnight Counts",
    labels={"overnight": "Overnight Count", "count": "Frequency"},
    marginal="box",
)
fig.update_layout(height=400)
fig.show()

## 4. Temporal Patterns

In [ ]:
# Daily average occupancy over time
daily_avg = df.groupby("date").agg(
    avg_occupancy=("occupancy_rate", "mean"),
    total_overnight=("overnight", "sum"),
    total_capacity=("capacity", "sum"),
).reset_index()

fig = px.line(
    daily_avg, x="date", y="avg_occupancy",
    title="Average Daily Occupancy Rate Over Time",
    labels={"date": "Date", "avg_occupancy": "Avg Occupancy Rate"},
)
fig.update_yaxes(tickformat=".0%")
fig.update_layout(height=400)
fig.show()

In [ ]:
# Total overnight stays over time
fig = px.line(
    daily_avg, x="date", y="total_overnight",
    title="Total Daily Overnight Stays Over Time",
    labels={"date": "Date", "total_overnight": "Total Overnight Stays"},
)
fig.update_layout(height=400)
fig.show()

In [ ]:
# Day-of-week patterns
dow_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dow_avg = df.groupby("day_of_week")["occupancy_rate"].mean().reset_index()
dow_avg["day_name"] = dow_avg["day_of_week"].map(lambda x: dow_labels[x])

fig = px.bar(
    dow_avg, x="day_name", y="occupancy_rate",
    title="Average Occupancy Rate by Day of Week",
    labels={"day_name": "Day of Week", "occupancy_rate": "Avg Occupancy Rate"},
)
fig.update_yaxes(tickformat=".0%")
fig.update_layout(height=400)
fig.show()

In [ ]:
# Monthly patterns
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
month_avg = df.groupby("month")["occupancy_rate"].mean().reset_index()
month_avg["month_name"] = month_avg["month"].map(lambda x: month_labels[x - 1])

fig = px.bar(
    month_avg, x="month_name", y="occupancy_rate",
    title="Average Occupancy Rate by Month",
    labels={"month_name": "Month", "occupancy_rate": "Avg Occupancy Rate"},
)
fig.update_yaxes(tickformat=".0%")
fig.update_layout(height=400)
fig.show()

In [ ]:
# Year-over-year trends
yearly_avg = df.groupby("year").agg(
    avg_occupancy=("occupancy_rate", "mean"),
    avg_overnight=("overnight", "mean"),
    avg_capacity=("capacity", "mean"),
).reset_index()

fig = px.bar(
    yearly_avg, x="year", y="avg_occupancy",
    title="Average Occupancy Rate by Year",
    labels={"year": "Year", "avg_occupancy": "Avg Occupancy Rate"},
)
fig.update_yaxes(tickformat=".0%")
fig.update_layout(height=400)
fig.show()

## 5. Shelter Type Comparison

In [ ]:
# How many shelter types?
print(f"Unique shelter types: {df['sheltertype'].nunique()}")
print(df["sheltertype"].value_counts())

In [ ]:
fig = px.box(
    df, x="sheltertype", y="occupancy_rate",
    title="Occupancy Rate Distribution by Shelter Type",
    labels={"sheltertype": "Shelter Type", "occupancy_rate": "Occupancy Rate"},
    color="sheltertype",
)
fig.update_yaxes(tickformat=".0%")
fig.update_layout(height=450, showlegend=False)
fig.show()

In [ ]:
# Occupancy over time by shelter type
type_monthly = (
    df.groupby([pd.Grouper(key="date", freq="MS"), "sheltertype"])
    ["occupancy_rate"]
    .mean()
    .reset_index()
)

fig = px.line(
    type_monthly, x="date", y="occupancy_rate", color="sheltertype",
    title="Monthly Avg Occupancy Rate by Shelter Type",
    labels={"date": "Date", "occupancy_rate": "Avg Occupancy Rate"},
)
fig.update_yaxes(tickformat=".0%")
fig.update_layout(height=450)
fig.show()

## 6. Individual Shelter Comparison

In [ ]:
# Shelter summary statistics
summary = compute_shelter_summary(df)
print(f"Total unique shelters: {len(summary)}")
summary.sort_values("mean_occupancy", ascending=False).head(15)

In [ ]:
# Top 10 shelters by average occupancy
top_shelters = summary.nlargest(10, "mean_occupancy")

fig = px.bar(
    top_shelters, x="mean_occupancy", y="shelter", orientation="h",
    title="Top 10 Shelters by Average Occupancy Rate",
    labels={"mean_occupancy": "Mean Occupancy Rate", "shelter": "Shelter"},
    color="mean_occupancy",
    color_continuous_scale="YlOrRd",
)
fig.update_xaxes(tickformat=".0%")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=450)
fig.show()

In [ ]:
# Capacity vs. Mean Occupancy scatter
fig = px.scatter(
    summary, x="mean_capacity", y="mean_occupancy",
    size="total_nights", color="sheltertype",
    hover_data=["shelter", "organization"],
    title="Shelter Capacity vs. Average Occupancy",
    labels={"mean_capacity": "Mean Capacity", "mean_occupancy": "Mean Occupancy Rate"},
)
fig.update_yaxes(tickformat=".0%")
fig.add_hline(y=0.9, line_dash="dash", line_color="red",
              annotation_text="90% threshold")
fig.update_layout(height=500)
fig.show()

## 7. Correlation Analysis

In [ ]:
# Add rolling features for correlation analysis
df_feat = add_rolling_features(df)

numeric_cols = [
    "capacity", "overnight", "occupancy_rate", "day_of_week", "month",
    "day_of_month", "rolling_7d_occupancy", "rolling_30d_occupancy",
    "lag_1d_occupancy", "lag_7d_occupancy",
]

corr = df_feat[numeric_cols].corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    title="Feature Correlation Matrix",
    aspect="auto",
)
fig.update_layout(height=550, width=700)
fig.show()

## 8. Key Takeaways

- **Occupancy rates** are often high, with a notable fraction of observations exceeding 90% or even 100% (overcapacity).
- **Temporal patterns** exist: occupancy varies by day of week, month, and year, suggesting seasonality is an important predictive feature.
- **Shelter type matters**: Different shelter categories exhibit distinct occupancy profiles.
- **Rolling and lag features** are strongly correlated with current occupancy, making them valuable predictors.
- **Capacity varies widely** across shelters, and the relationship between capacity and utilization is non-trivial.

These insights inform the feature engineering and modeling strategy in the prediction pipeline.